# Lab 3: Naive Bayes and Gradient Descent

Based on §4.4.4 (Naive Bayes) of James, Witten, Hastie, Tibshirani, and Taylor,
*An Introduction to Statistical Learning with Applications in Python* (Springer, 2023),
using the `Default` data from the `ISLP` package.

In Part 1, you'll build a naive Bayes classifier from scratch, using nothing but counting and
multiplying. In Part 2, you'll write gradient descent yourself.

### How this lab works

- Work through the parts in order.
- Everything you turn in is labeled by part number and letter, like **1(c)**, and marked as
  either **Code** or **Written**:
  - **Code:** write your code in the cell right below, which starts with `# 1(c)` and
    `# YOUR CODE HERE`. Many of these cells are partly written for you: fill in the part marked
    `# YOUR CODE HERE`, and leave the rest.
  - **Written:** replace `YOUR ANSWER HERE` in the **1(d) Answer:** cell right below. Written
    answers: 1–3 sentences.
- Most prompts have a collapsible **Hint**. Open it whenever you're stuck.
- The **Optional Naive Bayes Extras** at the end of the notebook are already written for you.
  Run them and read the explanations. There's nothing to turn in for them.
- Anything else, like "Run this cell," is just an instruction. There's nothing to turn in.

**Checklist** (17 required items):

| Part | Code | Written |
|---|---|---|
| 1 | 1(a), 1(b), 1(c), 1(e), 1(g), 1(h) | 1(d), 1(f), 1(i), 1(j), 1(k) |
| 2 | 2(a), 2(b), 2(d), 2(e) | 2(c), 2(f) |

- **Links to the reading.** Links like [Section 7](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-7) open the naive Bayes
  reading in your copy of the course repo (`Course_Documents/Slides/ch_4/`). They work on
  GitHub and in Jupyter or VS Code if you cloned the repo. On Colab, open the reading on GitHub
  instead.
- Before submitting, run **Restart & Run All** and make sure every cell runs.

**Note:** If you just run everything right now, you'll get some errors, because some of the
cells I've provided depend on cells you complete first. That's expected.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.naive_bayes import CategoricalNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_curve, roc_auc_score, log_loss
from ISLP import load_data

## Part 1: Naive Bayes from scratch

This part puts the naive Bayes classifier from the reading to work. We skip the derivation and
start from its result, equation 4.30 ([Section 10](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-10)), and build a classifier by counting.
If you haven't read it recently, its [summary](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#summary) is a good refresher.

### Part 1 Overview

Everything in Part 1 builds toward equation 4.30:

$$
\Pr(Y = k \mid X = x) \approx \frac{\pi_k \times f_{k1}(x_1) \times f_{k2}(x_2) \times \cdots \times f_{kp}(x_p)}{\sum_{l=1}^{K} \pi_l \times f_{l1}(x_1) \times f_{l2}(x_2) \times \cdots \times f_{lp}(x_p)}
$$

Here there are $K = 2$ classes (`No` and `Yes`) and $p = 3$ features.

1. Turn three columns of the data into yes-or-no features: a customer's values $x_1, x_2, x_3$.
2. Count the two ingredients: the **priors** $\pi_k$ (Step 1) and, for each class, the
   **share** of customers with each feature, $f_{kj}(x_j)$ (Step 2).
3. For one customer, multiply those into the **numerator**
   $\pi_k \times f_{k1}(x_1) \times f_{k2}(x_2) \times f_{k3}(x_3)$ for each class. The
   **denominator** is the sum of both classes' numerators, and dividing gives the posterior
   $\Pr(Y = \text{Yes} \mid X = x)$ (Step 3).
4. Wrap that calculation in a function (Step 4), compute $\Pr(Y = \text{Yes} \mid X = x)$ for
   every customer, and see what the model predicts.
5. Judge the model with precision, recall, and an ROC curve (Step 5).

The optional extras at the end of the notebook, already written for you, check your model
against scikit-learn, fix a failure caused by a share of 0, test the naive assumption itself,
and use `balance` as a number.

### The data: `Default`

Simulated data on 10,000 credit card customers. The columns:

| Column | Meaning |
|---|---|
| `default` | whether the customer defaulted on their credit card debt: `No` or `Yes` |
| `student` | whether the customer is a student: `No` or `Yes` |
| `balance` | the customer's average credit card balance, in dollars |
| `income` | the customer's income, in dollars |

We'll predict `default` from the other three.

In [ ]:
Default = load_data('Default')
display(Default.head())
Default['default'].value_counts()

Only 333 of the 10,000 customers defaulted, about 3.3%. Defaulting is rare, which will matter
later.

### Three yes-or-no features

The reading builds naive Bayes by **counting**: for each class, what share of its customers has
each feature? Counting works on yes-or-no features, so we'll turn each column into one:

| Feature | Definition |
|---|---|
| `student` | the customer is a student |
| `high_balance` | the customer's balance is over \$1,500 |
| `high_income` | the customer's income is over \$30,000 |

Run this cell to build a table with the class and two of the features. Each feature column holds
`True` or `False`.

<details>
<summary>Code explanation</summary>

- `pd.DataFrame({...})` builds a table from a dictionary: each key becomes a column name, and
  each value (a Series) becomes that column's values.
- A comparison on a whole column, like `Default['income'] > 30000`, checks every row at once
  and returns a Series of `True`/`False`, one per customer. No loop is needed.
- `.astype(str)` turns the `default` column from pandas' `category` type into plain text
  (`'No'`/`'Yes'`), which keeps later group-by and table output simple.
- `data.head()` shows the first 5 rows.

</details>

In [ ]:
data = pd.DataFrame({
    'default': Default['default'].astype(str),       # 'No' or 'Yes'
    'student': Default['student'] == 'Yes',
    'high_income': Default['income'] > 30000,
})
data.head()

**1(a) · Code.** Add the third feature: a column of `data` named `high_balance` that is `True`
when the customer's `balance` (in `Default`) is over 1,500.

<details>
<summary>Hint</summary>

`data` already exists, so you can't add to it with the dictionary from the cell above. You need
two pieces:

- **The values.** Only the right side of the `high_income` line carries over:
  `Default['income'] > 30000` gives a column of `True` and `False`. Do the same with the
  `balance` column and 1500.
- **The new column.** Add a column to an existing DataFrame by assigning to it:
  `data['new_column'] = values`.

</details>

In [ ]:
# 1(a)
### YOUR CODE HERE
data.head()

Run this cell to see how each feature splits across the two classes. Each table counts the
customers in each combination of `default` (rows) and the feature (columns), with totals.

In [ ]:
features = ['student', 'high_balance', 'high_income']

for feature in features:
    display(pd.crosstab(data['default'], data[feature], margins=True))

### Step 1: The priors

The **prior** $\pi_k$ is the share of all customers who are in class $k$, regardless of their other feature values ([Section 9](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-9)). Here there are two classes, `No` and `Yes`.

**1(b) · Code.** Compute the priors, and name the result `priors`. It should be a Series indexed
by class (`'No'` and `'Yes'`), holding each class's share of the 10,000 customers.

<details>
<summary>Hint</summary>

`value_counts()` counts how many times each value appears in a column. Passing
`normalize=True` divides those counts by the total, which turns them into shares.

</details>

In [ ]:
# 1(b)
### YOUR CODE HERE
priors

### Step 2: The one-feature shares

Naive Bayes needs, for each class and each feature, the **share of that class with that
feature**. ISL writes it as $f_{kj}(x_j)$: the share of class $k$ whose feature $j$ equals
$x_j$ ([Section 9](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-9)). The naive assumption is what lets us count each feature on
its own, among the whole class ([Section 7](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-7)).

For a yes-or-no feature, we only need to store the share with "yes". The share with "no" is
1 minus that. This shortcut works because each feature has only two values. A feature with more values needs one share per value. More classes need no
shortcut at all: equation 4.30 just gets one term per class ([Section 10](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-10)).

Here is the worked example for `student`, written out the long way first. For each class:

- `data[data['default'] == 'Yes']` keeps only the rows where the condition is `True`, here the
  defaulters.
- `.sum()` on a `True`/`False` column counts the `True`s, because Python counts `True` as 1 and
  `False` as 0. So `defaulters['student'].sum()` is the number of defaulters who are students.
- `len(...)` is the number of rows, so dividing gives the share.

In [ ]:
defaulters = data[data['default'] == 'Yes']
non_defaulters = data[data['default'] == 'No']

students_yes = defaulters['student'].sum() / len(defaulters)
students_no = non_defaulters['student'].sum() / len(non_defaulters)

print(f'share of defaulters who are students:     {students_yes:.6f}')
print(f'share of non-defaulters who are students: {students_no:.6f}')

Now, we'll see how this is usually done with more concise code:

- `groupby('default')` splits the customers into the two classes.
- `['student']` picks the column.
- `.mean()` averages it within each class. The average of a `True`/`False` column is the count of
  `True`s divided by the number of rows, which is exactly the share computed above.

The result is a Series with one share per class, labeled `No` and `Yes`. We'll use this
version from here on, because those labels let us look up a class's share by name.

In [ ]:
student_share = data.groupby('default')['student'].mean()
student_share

So 38.1% of defaulters are students, and 29.1% of non-defaulters are.

**1(c) · Code.** Compute the same shares for the other two features. Name them
`high_balance_share` and `high_income_share`.

<details>
<summary>Hint</summary>

Copy the `student_share` line, and change only the column name.

`groupby('default')` splits the customers into the two classes. Selecting a column and calling
`.mean()` then gives that column's average within each class.

</details>

In [ ]:
# 1(c)
### YOUR CODE HERE


Run this cell to collect all three shares into one table. Rows are classes, and columns are
features. Each entry is the share of that class with that feature.

In [ ]:
shares = pd.DataFrame({
    'student': student_share,
    'high_balance': high_balance_share,
    'high_income': high_income_share,
})
shares

**1(d) · Written.** Compare the two rows of `shares`. Which feature differs the most between
defaulters and non-defaulters? Which one barely differs? Which of the two do you expect to
matter more when we classify a customer, and why?

**1(d) Answer:**

_YOUR ANSWER HERE_


### Step 3: Score one customer by hand

Equation 4.30 ([Section 10](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-10)) puts the pieces together. For each class $k$, the
**numerator** is the prior times one share per feature:

$$
\pi_k \times f_{k1}(x_1) \times f_{k2}(x_2) \times f_{k3}(x_3)
$$

The **posterior** for a class is its numerator divided by the sum of both classes' numerators
([Section 8](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-8)):

$$
\Pr(Y = \text{Yes} \mid x) = \frac{\text{numerator for Yes}}{\text{numerator for Yes} + \text{numerator for No}}
$$

Each $f_{kj}(x_j)$ must match **this customer's** value of feature $j$. If the customer has the
feature, use the share from the `shares` table. If not, use 1 minus it.

Take a new customer who is **not a student**, has a **high balance**, and has a **high income**.
For the defaulters, the numerator is

$$
\underbrace{\pi_{\text{Yes}}}_{\text{prior}} \times
\underbrace{(1 - 0.381)}_{\text{not a student}} \times
\underbrace{0.775}_{\text{high balance}} \times
\underbrace{0.526}_{\text{high income}}
$$

**1(e) · Code.** The cell below computes the numerator for `Yes` for you. Define 
`numerator_no` similarly, using the `No` row of `shares`, and then define `posterior_yes`.

<details>
<summary>Hint</summary>

- `shares.loc['No', 'student']` is the share of non-defaulters who are students.
- This customer is *not* a student, so use 1 minus that share. They *do* have the other two
  features, so use those shares as they are.
- The posterior divides the `Yes` numerator by the sum of both numerators.

</details>

In [ ]:
# 1(e)
numerator_yes = (priors['Yes']
                 * (1 - shares.loc['Yes', 'student'])    # not a student
                 * shares.loc['Yes', 'high_balance']     # high balance
                 * shares.loc['Yes', 'high_income'])     # high income

### YOUR CODE HERE


Run this cell to see your results.

In [ ]:
print(f'numerator for Yes: {numerator_yes:.5f}')
print(f'numerator for No:  {numerator_no:.5f}')
print(f'posterior probability of default: {posterior_yes:.3f}')

**1(f) · Written.** Before we knew anything about this customer, their chance of defaulting was
the prior, about 3.3%. Which of their three features is mostly responsible for raising it to
about 23%? Even so, the posterior is still well under 50%. Why?

**1(f) Answer:**

_YOUR ANSWER HERE_


### Step 4: A function that scores any customer

To classify every customer, we wrap Step 3 in a function. It loops over the classes and, inside
that, over the features, multiplying one share into the numerator each time.

**1(g) · Code.** Complete `naive_bayes_posterior`. Inside the inner loop, multiply `numerator` by
the right share for this customer: `share_yes` if they have the feature, and `1 - share_yes` if
they don't. Leave the rest of the function as it is.

`customer` can be a dictionary like `{'student': False, 'high_balance': True, ...}` or one row
of `data`. Either way, `customer[feature]` is `True` or `False`.

<details>
<summary>Hint</summary>

- `numerator *= x` is short for `numerator = numerator * x`.
- An `if`/`else` works fine. A one-line version is
  `value_a if condition else value_b`, which gives `value_a` when `condition` is `True`.

</details>

In [ ]:
# 1(g)
def naive_bayes_posterior(customer, priors, shares):
    # Return Pr(default = Yes | customer's features), using equation 4.30.
    numerators = {}
    for k in priors.index:                              # each class: 'No', then 'Yes'
        numerator = priors[k]                           # start from the prior
        for feature in shares.columns:                  # one share per feature
            share_yes = shares.loc[k, feature]          # share of class k with this feature
            ### YOUR CODE HERE
        numerators[k] = numerator
    return numerators['Yes'] / sum(numerators.values())

Run this cell to check your function on the customer from Step 3. The two numbers should match.

In [ ]:
new_customer = {'student': False, 'high_balance': True, 'high_income': True}

print(f'from your function: {naive_bayes_posterior(new_customer, priors, shares):.5f}')
print(f'from 1(e):          {posterior_yes:.5f}')

**1(h) · Code.** Score every customer in `data`.

1. Compute `posterior`: a Series with one posterior probability of default per customer.
2. Compute `prediction`: a Series that is `'Yes'` where `posterior` is at least 0.5, and `'No'`
   everywhere else.

<details>
<summary>Hint</summary>

- `data[features].apply(f, axis=1)` calls `f` once per row and collects the results into a
  Series. Here `f` needs to call your function with that row, so use a `lambda`:
  `lambda customer: naive_bayes_posterior(customer, priors, shares)`.
- `np.where(condition, 'Yes', 'No')` picks `'Yes'` where the condition is `True`. Wrap it in
  `pd.Series(..., index=data.index)` to get a Series.

</details>

In [ ]:
# 1(h)
### YOUR CODE HERE


Run this cell. With three yes-or-no features, there are only eight possible customers, so the
model can only give eight different answers. The table shows each combination that appears in
the data, its posterior, and how many customers have it. Only seven appear: no customer is a
student with both a high balance and a high income.

In [ ]:
summary = (data.assign(posterior=posterior)
               .groupby(features)['posterior']
               .agg(posterior='first', customers='size'))
display(summary)
print('Customers predicted to default:', (prediction == 'Yes').sum())

Notice the last line: at a threshold of 0.5, the model predicts that **nobody** defaults. Even
the riskiest combination gets a posterior of only about 0.38. Step 5 comes back to this.

**1(i) · Written.** With these three features, the naive assumption let us estimate just one
share per class per feature, each counted among the whole class. What would we have needed to
count without it, and why does that get much worse as we add features?

**1(i) Answer:**

_YOUR ANSWER HERE_


### Step 5: How good is it?

In [Notebook 2](../../Course_Documents/Slides/ch_4/ch_4_demos_2_classification_metrics.ipynb) (Sections 4–8), we judged a classifier with a confusion matrix, recall,
precision, and an ROC curve. scikit-learn computes all of them. Run this cell to see the first
three for our model, at the usual threshold of 0.5 and at a lower threshold of 0.2.

<details>
<summary>Code explanation</summary>

- `np.where(posterior >= threshold, 'Yes', 'No')` turns each posterior into a prediction.
- `confusion_matrix(actual, predicted, labels=['No', 'Yes'])` counts the four outcomes.
  `labels` fixes the order of the rows and columns, `No` first.
- `recall_score` and `precision_score` need `pos_label='Yes'`. Our classes are text, so
  scikit-learn has to be told which one counts as positive.
- `zero_division=0` reports precision as 0 when nobody is flagged, instead of warning that it's
  0/0.

</details>

In [ ]:
def evaluate(threshold):
    predicted = np.where(posterior >= threshold, 'Yes', 'No')
    table = confusion_matrix(data['default'], predicted, labels=['No', 'Yes'])
    display(pd.DataFrame(table, index=['actual No', 'actual Yes'],
                         columns=['predicted No', 'predicted Yes']))
    recall = recall_score(data['default'], predicted, pos_label='Yes')
    precision = precision_score(data['default'], predicted, pos_label='Yes', zero_division=0)
    print(f'threshold {threshold}: recall = {recall:.3f}, precision = {precision:.3f}')

evaluate(0.5)
evaluate(0.2)

At 0.5, the model flags no one, so its recall is 0: it catches none of the 333 defaulters. That's
the rare-class problem from Notebook 2. A customer's posterior has to overcome a prior of only
3.3%, and with these three features, none get past about 0.38.

At 0.2, it flags exactly the 911 customers with a high balance.

**1(j) · Written.** Look at the results for a threshold of 0.2. In terms of customers, what do
its recall and its precision mean? Which threshold, 0.5 or 0.2, is more useful to a bank that
wants to catch defaulters, and what does it cost?

<details>
<summary>Hint</summary>

- Recall: of the customers who actually defaulted, what share did we flag?
- Precision: of the customers we flagged, what share actually defaulted?
- Use the confusion matrix to turn the shares into numbers of customers.

For example, suppose a different model's confusion matrix were

| | predicted No | predicted Yes |
|---|---|---|
| **actual No** | 9,500 | 160 |
| **actual Yes** | 60 | 40 |

- **Recall** uses the `actual Yes` row: $40 / (60 + 40) = 0.4$. In customers: of the 100 who
  defaulted, we flagged 40 and missed 60.
- **Precision** uses the `predicted Yes` column: $40 / (160 + 40) = 0.2$. In customers: of the
  200 we flagged, 40 defaulted, and the other 160 were false alarms.

Do the same with the real confusion matrix for a threshold of 0.2.

</details>

**1(j) Answer:**

_YOUR ANSWER HERE_


Recall and precision judge one threshold at a time. An **ROC curve** shows every threshold at
once. Each threshold gives one point:

- horizontal axis = **false positive rate** (the share of non-defaulters we flag)
- vertical axis = **recall** (the share of defaulters we catch)

**AUC**, the area under the curve, is the chance that a randomly chosen defaulter got a higher
posterior than a randomly chosen non-defaulter. An AUC of 0.5 is a coin flip, and 1.0 is
perfect (Notebook 2, Section 8).

<details>
<summary>Code explanation</summary>

- `roc_curve(actual, scores, pos_label='Yes')` tries every threshold and returns three arrays:
  the false positive rates, the recalls (scikit-learn calls them true positive rates), and the
  thresholds.
- `roc_auc_score(actual == 'Yes', scores)` returns the AUC. It wants the actual classes as
  `True`/`False`, so we compare with `'Yes'` first.
- The false positive rate and recall at 0.2 are counted directly, to mark that point.

</details>

In [ ]:
false_positive_rate, recall, thresholds = roc_curve(data['default'], posterior, pos_label='Yes')
auc = roc_auc_score(data['default'] == 'Yes', posterior)

flagged = posterior >= 0.2
is_yes = data['default'] == 'Yes'
fpr_at_02 = (flagged & ~is_yes).sum() / (~is_yes).sum()
recall_at_02 = (flagged & is_yes).sum() / is_yes.sum()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(false_positive_rate, recall, color='black', lw=2.5, marker='o',
        label=f'naive Bayes (AUC = {auc:.2f})')
ax.plot([0, 1], [0, 1], color='0.6', ls='--', label='random guessing')
ax.plot(fpr_at_02, recall_at_02, 'o', ms=13, color='#D55E00', label='threshold 0.2')
ax.set_xlabel('false positive rate (share of non-defaulters flagged)')
ax.set_ylabel('recall (share of defaulters caught)')
ax.set_title('ROC curve')
ax.legend(loc='lower right')
plt.show()

The curve has only a few corners. Our model gives just seven different posteriors, one per
combination in the summary table after 1(h). Lowering the threshold flags a whole combination
at once, so the curve jumps from corner to corner. The orange point, a threshold of 0.2,
catches 77.5% of the defaulters while flagging 6.8% of the non-defaulters.

**1(k) · Written.** Our AUC is about 0.85. In Notebook 2, logistic regression on `balance` had
an AUC of about 0.95. Why might our naive Bayes model rank customers less well?

<details>
<summary>Hint</summary>

Take two customers who match on `student` and `high_income`, with balances of \$1,600 and
\$2,600. What posterior does our model give each of them? What would logistic regression on
`balance` do?

</details>

**1(k) Answer:**

_YOUR ANSWER HERE_


*End of Part 1.* The **Optional Naive Bayes Extras** come after Part 2. They're already
written for you. Scroll down to them to check your model against scikit-learn, fix the failure
caused by a share of 0, compare naive Bayes with exact Bayes, and use `balance` as a number.

## Part 2: Gradient Descent

In Section 10 of the [gradient descent notebook](../../Course_Documents/Slides/ch_4/gradient_descent_demos.ipynb), gradient descent fit a logistic
regression on `balance`. Now you'll write that loop yourself, fit a model with two features,
`balance` and `income`, and see why the scale of the features matters.

### Part 2 Overview

1. Run one step of gradient descent, piece by piece. The matrix code is handed to you.
2. Put those pieces into a loop with a stopping rule (2(a)).
3. Fit the model on the raw features, and try bigger learning rates (2(b), 2(c)).
4. Find out why it's so slow, standardize the features (2(d)), and fit again (2(e), 2(f)).

### The model

For customer $i$, with balance $x_{i1}$ and income $x_{i2}$, the model's probability of
default is

$$
p_i = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2})}}
$$

The loss is the average negative log-likelihood from Section 10:

$$
L(\beta) = -\frac{1}{n}\sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \Big]
$$

Each step of gradient descent updates all three coefficients at once, where the gradient
$\nabla L$ holds one slope per coefficient:

$$
\beta \leftarrow \beta - \eta \, \nabla L(\beta)
$$

Run this cell to set up the data. `y01` is 1 for customers who defaulted and 0 for the rest.
`X_raw` has one row per customer and one column per coefficient: a column of 1s for the
intercept (Section 10 explains why), then balance and income.

In [ ]:
y01 = (Default['default'] == 'Yes').to_numpy().astype(int)    # 1 = defaulted, 0 = didn't
raw = Default[['balance', 'income']].to_numpy()
X_raw = np.column_stack([np.ones(len(raw)), raw])

print('X_raw shape:', X_raw.shape)
pd.DataFrame(X_raw[:5], columns=['1s', 'balance', 'income'])

### One step, piece by piece

Every step of gradient descent does the same four things. This cell does one step, starting
from $\beta = (0, 0, 0)$, with a learning rate of $\eta = 10^{-8}$ (`1e-8` in Python). Study the code and try to understand it.

The `@` symbol means matrix multiplication. The
[Code Explanations in Section 10 of the gradient descent notebook](../../Course_Documents/Slides/ch_4/gradient_descent_demos.ipynb#code-explanations-10)
break down the matrix algebra behind `X_raw @ beta` and the gradient line step by step, with the
shape of each array. They'll help you understand the matrix multiplications in this code.

<details>
<summary>Code explanation</summary>

1. `X_raw @ beta` computes $\beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2}$ for every customer at
   once, and `p` turns each one into a probability. `p` has shape `(10000,)`: one per customer.
2. `log_loss(y01, p)` is the loss $L(\beta)$ above.
3. `X_raw.T @ (p - y01) / n` is the gradient $\nabla L$: one slope per coefficient, shape
   `(3,)`. Section 10's **Code Explanations** break this line down.
4. The update rule moves every coefficient downhill, by $\eta$ times its slope.

</details>

In [ ]:
n = len(y01)
beta = np.zeros(3)                         # (β₀, β₁, β₂) = (0, 0, 0)
eta = 1e-8

p = 1 / (1 + np.exp(-(X_raw @ beta)))      # 1. a probability for every customer
loss = log_loss(y01, p)                    # 2. the loss - we imported this
grad = X_raw.T @ (p - y01) / n             # 3. the gradient: one slope per coefficient
beta = beta - eta * grad                   # 4. the update rule

print('loss at the start:', round(loss, 4))
print('gradient:         ', grad)
print('beta after 1 step:', beta)

**2(a) · Code.** Complete `gradient_descent`. It repeats the step above, up to `max_steps`
times. Piece 1 and the stopping rule are written for you. Fill in the two places marked
`YOUR CODE HERE`:

1. Above the stopping rule: compute the loss (piece 2), and append it to `losses`.
2. Below the stopping rule: compute the gradient and update `beta` (pieces 3 and 4).

Use the function's own names, `X`, `y`, and `eta`, not `X_raw` and `y01`.

**The stopping rule.** It stops the loop once the loss changes by less than `tol` from one step
to the next, because then more steps would barely help (Section 7 of the [gradient descent notebook](../../Course_Documents/Slides/ch_4/gradient_descent_demos.ipynb)).
`losses[-1]` is the newest loss, and `losses[-2]` is the one before it. `break` exits the loop
right away.

<details>
<summary>Hint</summary>

- Each line you need is in the cell above, with `X_raw` changed to `X` and `y01` changed to
  `y`. Use `len(y)` for $n$.
- `losses.append(value)` adds a value to the end of the list.

</details>

In [ ]:
# 2(a)
def gradient_descent(X, y, eta, max_steps=5000, tol=1e-7):
    # Return the final beta and the list of losses, one per step.
    beta = np.zeros(X.shape[1])                    # start every coefficient at 0
    losses = []
    for step in range(max_steps):
        p = 1 / (1 + np.exp(-(X @ beta)))          # piece 1
        # Update the list of losses
        ### YOUR CODE HERE

        if len(losses) > 1 and abs(losses[-2] - losses[-1]) < tol:    # stopping rule
            break

        # Update the model parameters
        ### YOUR CODE HERE
    return beta, losses

Run this cell to check your function. One step from 0 should match the cell before 2(a).

In [ ]:
beta_check, losses_check = gradient_descent(X_raw, y01, eta=1e-8, max_steps=1)

print('first loss matches:', np.isclose(losses_check[0], loss))
print('beta matches:      ', np.allclose(beta_check, beta))

### Fitting the raw features

Run this cell to fit the model on `X_raw` with $\eta = 10^{-8}$. It takes about 10 seconds. For
reference, it also fits the same model with scikit-learn, which uses a faster method, to get
the lowest loss possible. `plot_losses` draws loss curves like the ones in Section 7.

In [ ]:
def plot_losses(runs, log_steps=False):
    # runs: a dictionary of {label: list of losses}
    fig, ax = plt.subplots(figsize=(7, 4))
    for (label, losses), color in zip(runs.items(), ['black', '#0072B2', '#CC79A7']):
        ax.plot(np.arange(1, len(losses) + 1), losses, color=color, lw=2, label=label)
    ax.axhline(best_loss, color='#D55E00', ls='--', label='best possible (scikit-learn)')
    ax.set_xlabel('step')
    ax.set_ylabel('loss')
    ax.set_ylim(0, None)
    if log_steps:
        ax.set_xscale('log')     # spreads out the first few steps
    ax.legend()
    plt.show()


sk_raw = LogisticRegression(C=np.inf).fit(raw, y01)     # C=np.inf: plain maximum likelihood, like ours
best_loss = log_loss(y01, sk_raw.predict_proba(raw)[:, 1])

beta_raw, losses_raw = gradient_descent(X_raw, y01, eta=1e-8)

print(f'steps taken: {len(losses_raw)}')
print(f'final loss:  {losses_raw[-1]:.4f}')
print(f'best loss:   {best_loss:.4f}  (scikit-learn)')
plot_losses({'η = 1e-8': losses_raw})

The loss drops fast on the first step, and then crawls. The loop stopped after 4,775 steps
because the loss was changing by less than `tol` per step. But the final loss, about 0.174,
is more than twice the best possible, 0.079. **The stopping rule only tells us that the loss
stopped changing much. It doesn't tell us that we reached the bottom.**

**2(b) · Code.** Maybe the steps are just too small. Fit `X_raw` two more times, with bigger
learning rates:

- `eta=1e-7` (10 times bigger), saving the results as `beta_big, losses_big`
- `eta=1e-6` (100 times bigger), saving the results as `beta_huge, losses_huge`

It takes about 10 seconds. Expect a `RuntimeWarning` about an overflow.

<details>
<summary>Hint</summary>

Copy the `gradient_descent` line from the cell above, and change the names and `eta`.

</details>

In [ ]:
# 2(b)
### YOUR CODE HERE


Run this cell to compare all three runs. The steps are on a log scale (1, 10, 100, ...), so
you can see what happens in the first few.

In [ ]:
for label, losses in [('η = 1e-7', losses_big), ('η = 1e-6', losses_huge)]:
    print(f'{label}: {len(losses)} steps, final loss {losses[-1]:.4f}')

plot_losses({'η = 1e-8': losses_raw, 'η = 1e-7': losses_big, 'η = 1e-6': losses_huge}, log_steps=True)

The overflow warning means `np.exp` got a number too big to store. The steps were so big that
some probabilities came out as exactly 0 or 1.

**2(c) · Written.** What happened to the loss with each bigger learning rate? Did either one get
closer to the best possible loss?

**2(c) Answer:**

_YOUR ANSWER HERE_


### Why is it so slow? Look at the features

Run this cell to see the features' ranges, the gradient from the first step, and the
coefficients we found next to the best ones.

In [ ]:
display(Default[['balance', 'income']].describe().round(0))

print('gradient at the first step:')
print(pd.Series(grad, index=['intercept', 'balance', 'income']).round(4))

pd.DataFrame([beta_raw, [sk_raw.intercept_[0], *sk_raw.coef_[0]]],
             index=['gradient descent, η = 1e-8', 'scikit-learn (best)'],
             columns=['intercept', 'balance', 'income'])

Income is measured in tens of thousands of dollars, balance in hundreds, and the intercept's
column is all 1s. The gradient follows suit: at the first step, income's slope is about
15,690, balance's is about 359, and the intercept's is about 0.47.

One learning rate moves every coefficient, by $\eta$ times its slope. Income's slope is so big
that $\eta$ has to be tiny to keep income's steps from overshooting. With that same tiny
$\eta$, the intercept moves only about $10^{-8} \times 0.47 \approx 0.000000005$ per step. It
needs to get to about $-11.5$, and after 4,775 steps it has only reached about $-0.000001$.
Make $\eta$ bigger, and income's steps overshoot, as you saw in 2(b).

On a contour plot, this is the long, narrow valley from Section 9 of the gradient descent
notebook (Demo 9b): steep across, and nearly flat along the bottom.

### Standardizing

The fix is to put every feature on the same scale. To **standardize** a feature, subtract its
mean and divide by its standard deviation. A standardized value says how many standard
deviations a customer is above the average. For example, a standardized balance of 2 means
about \$967 above the average balance, since balance's standard deviation is about \$484.
Section 10 used standardized balance for exactly this reason.

**2(d) · Code.** Compute `raw_std`: each column of `raw`, minus that column's mean, divided by
that column's standard deviation.

<details>
<summary>Hint</summary>

- `raw.mean(axis=0)` gives the mean of each column, and `raw.std(axis=0)` gives each column's
  standard deviation. (`axis=0` means "down the rows," so you get one number per column.)
- `raw - raw.mean(axis=0)` subtracts each column's own mean from every row. NumPy lines them up
  by column.

</details>

In [ ]:
# 2(d)
### YOUR CODE HERE
X_std = np.column_stack([np.ones(len(raw_std)), raw_std])     # the column of 1s stays 1s
pd.DataFrame(raw_std, columns=['balance', 'income']).describe().round(2)

Both columns now have a mean of 0 and a standard deviation of 1, and similar ranges.

**2(e) · Code.** Fit a logistic regression model to the standardized data,`X_std`, with `eta=4`, saving the results as `beta_std, losses_std`, using your `gradient_descent` function. That's
400 million times the $\eta = 10^{-8}$ we started with.

<details>
<summary>Hint</summary>

Same as 2(b), with `X_std` in place of `X_raw`.

</details>

In [ ]:
# 2(e)
### YOUR CODE HERE


Run this cell to see the loss curve, and to compare the coefficients with scikit-learn's, fit
on the same standardized features.

In [ ]:
print(f'steps taken: {len(losses_std)}')
print(f'final loss:  {losses_std[-1]:.4f}')
print(f'best loss:   {best_loss:.4f}  (scikit-learn)')
plot_losses({'standardized, η = 4': losses_std})

sk_std = LogisticRegression(C=np.inf).fit(raw_std, y01)
pd.DataFrame([beta_std, [sk_std.intercept_[0], *sk_std.coef_[0]]],
             index=['gradient descent', 'scikit-learn'],
             columns=['intercept', 'balance', 'income']).round(3)

This time the loop stops after 364 steps, within 0.0001 of the best possible loss, and the
coefficients nearly match scikit-learn's.

These coefficients are **per standard deviation**. A balance coefficient of 2.70 means that one
more standard deviation of balance (about \$484) raises the log-odds of default by 2.70. That's
why they don't match `sk_raw`'s coefficients, which are per dollar, even though both models make
essentially the same predictions.

**2(f) · Written.** Why did standardizing fix the problem, when a bigger learning rate couldn't?

**2(f) Answer:**

_YOUR ANSWER HERE_


*End of Part 2.* The optional naive Bayes extras from Part 1 follow.

## Optional Naive Bayes Extras

These are already written for you. Run each cell and read the explanation. There's nothing to
turn in. These cells use your `posterior` and your function from Part 1, so finish 1(g) and
1(h) first.

### Extra A: Check your model against scikit-learn

scikit-learn has naive Bayes built in. For yes-or-no and other category features, it's
`CategoricalNB`. It expects numbers, so we pass each feature as 1 (yes) or 0 (no). Its `alpha`
setting controls smoothing, which Extra B explains. A tiny `alpha` turns smoothing off, so it
should match what you computed.

In [ ]:
X = data[features].astype(int)                # True/False becomes 1/0
sk_model = CategoricalNB(alpha=1e-10)         # alpha near 0: no smoothing
sk_model.fit(X, data['default'])

yes_column = list(sk_model.classes_).index('Yes')
sk_posterior = sk_model.predict_proba(X)[:, yes_column]
print('Largest difference from your posteriors:', np.abs(sk_posterior - posterior).max())

The difference is essentially 0 (it's rounding error). The library does exactly what you just
did: priors, one share per class and feature, multiply, and divide by the sum.

### Extra B: When a share is exactly 0, and add-one smoothing

So far every share has rested on hundreds of customers. Here is a failure mode that a share can
cause, and the standard fix ([Sections 6](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-6) and [7](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-7)).

We'll add a fourth feature, `zero_balance`: the customer's balance is exactly \$0. **We chose
this feature to demonstrate the problem.** In this data, 499 customers have a zero balance, and
none of them defaulted.

In [ ]:
data['zero_balance'] = Default['balance'] == 0
features4 = features + ['zero_balance']

shares4 = data.groupby('default')[features4].mean()   # the 1(c) calculation, all four at once
shares4

The share of defaulters with a zero balance is exactly 0. Now score a customer who has a zero
balance (not a student, no high balance, high income):

In [ ]:
zero_customer = {'student': False, 'high_balance': False, 'high_income': True, 'zero_balance': True}

print(f'posterior with 3 features (ignoring zero_balance): {naive_bayes_posterior(zero_customer, priors, shares):.5f}')
print(f'posterior with all 4 features:                    {naive_bayes_posterior(zero_customer, priors, shares4):.5f}')

With the fourth feature, the posterior is **exactly 0**. One zero share makes the whole `Yes`
numerator 0, no matter what the other features say. The model is now *certain* this customer
won't default, based on 0 out of 333 defaulters. That's too strong a conclusion from a sample:
a rare event that didn't happen among these 333 people could still happen to the next one. And
a certainty can't be changed by any other evidence.

**The fix: add-one smoothing.** Before dividing, pretend each class has one extra customer with
the feature and one extra without it. For a yes-or-no feature, that's

$$
\text{smoothed share} = \frac{\text{count with the feature} + 1}{\text{class size} + 2}
$$

Each piece comes from those two imaginary customers:

- **The $+1$ on top** is the imaginary customer *with* the feature, added to the real count.
- **The $+2$ on the bottom** is the class size growing by both imaginary customers: one with the
  feature and one without. A share is always out of everyone in the class, and the class now has
  two more people in it.

We add one of *each* so that the share without the feature gets the same treatment:
$\frac{\text{count without} + 1}{\text{class size} + 2}$. The two smoothed shares still add to 1,
just as before. If we added only a customer with the feature, every share would be nudged up.

For `zero_balance` among the 333 defaulters:

$$
\text{with: } \frac{0 + 1}{333 + 2} = \frac{1}{335} \approx 0.003
\qquad
\text{without: } \frac{333 + 1}{333 + 2} = \frac{334}{335} \approx 0.997
$$

No share can be 0 or 1 anymore: the top is at least 1, and it's always smaller than the
bottom.

In [ ]:
counts = data.groupby('default')[features4].sum()     # customers with each feature, per class
class_sizes = data['default'].value_counts()
smoothed_shares = (counts + 1).div(class_sizes + 2, axis=0)

display(shares4.round(4))
display(smoothed_shares.round(4))

Compare the two tables. The shares that rest on a whole class barely move: the defaulters'
high-balance share goes from 0.7748 to 0.7731. The zero share becomes $1/335 \approx 0.003$,
small but not impossible. Now the zero-balance customer gets a small posterior instead of a
certain 0:

In [ ]:
print(f'posterior with smoothed shares: {naive_bayes_posterior(zero_customer, priors, smoothed_shares):.5f}')

sk_smoothed = CategoricalNB(alpha=1).fit(data[features4].astype(int), data['default'])
print('scikit-learn with alpha=1 (its default) gives the same smoothed posterior:',
      f"{sk_smoothed.predict_proba(pd.DataFrame([zero_customer]).astype(int))[0, 1]:.5f}")

scikit-learn's default, `alpha=1`, is exactly this add-one smoothing.

### Extra C: Exact Bayes versus naive Bayes

The naive assumption says that within each class, the features are unrelated: knowing one tells
you nothing about another ([Section 7](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-7)). With only three features and 10,000
customers, we can also do **exact** Bayes, by counting the customers in each of the 8 cells
directly ([Section 6](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-6)). Then the exact posterior for a cell is just the share of
its customers who defaulted. Here are both, side by side:

In [ ]:
cells = data.groupby(features + ['default']).size().unstack(fill_value=0)
every_cell = pd.MultiIndex.from_product([[False, True]] * 3, names=features)
cells = cells.reindex(every_cell, fill_value=0)       # include cells with no customers

cells['exact'] = cells['Yes'] / (cells['Yes'] + cells['No'])
cells['naive'] = [naive_bayes_posterior(dict(zip(features, cell)), priors, shares)
                  for cell in cells.index]
cells.round(3)

Three things stand out.

1. **An empty cell.** No customer at all is a student with a high balance and a high income, so
   the exact posterior there is $0/0$ (shown as `NaN`). Naive Bayes still gives an answer, about
   0.31, built from shares that each rest on a whole class.
2. **A tiny cell.** Only 9 customers are students with a high income and no high balance, and 1
   of them defaulted. The exact posterior, $1/9 \approx 0.11$, rests on 9 people, so it's
   unreliable: one more defaulter would nearly double it.
3. **Naive and exact disagree even in big cells.** For example, among the 423 non-students with
   a high balance and a high income, the exact posterior is about 0.30 and the naive one is about
   0.23. That's the cost of the naive assumption, which isn't true here. Run the next cell to
   see why.

In [ ]:
pd.crosstab([data['default'], data['student']], data['high_income'])

Within **each** class, students almost never have a high income: 8 of 2,817 non-defaulting
students do, and 1 of 127 defaulting students. Among non-students, about 84% do. So within a
class, knowing that a customer is a student tells you a lot about their income, which is exactly
what the naive assumption says can't happen.

<details>
<summary>Think about it: if the assumption is false here, why use naive Bayes at all?</summary>

Exact counting needs every cell to hold plenty of customers, and that fails quickly: with 20
yes-or-no features, there are over a million cells per class. Naive Bayes always counts each
feature among a whole class, so its shares stay reliable however many features there are. It
accepts some error from a false assumption (bias) in exchange for estimates that don't swing
wildly from one sample to the next (low variance). Here, the bias moves some posteriors by a few
percentage points.

</details>

### Extra D: A numeric feature

Turning `balance` into yes-or-no throws information away: a balance of \$1,501 and one of
\$2,600 both count as "high". Naive Bayes can use the number itself. For a numeric feature, a
share doesn't make sense, since almost no customer has *exactly* a given balance. Instead,
$f_{kj}(x_j)$ measures how typical the value $x_j$ is in class $k$. A common choice is a bell
curve fitted to that feature within each class, using the class's mean and standard deviation
([Section 9](../../Course_Documents/Slides/ch_4/naive-bayes-reading/naive-bayes-reading.md#section-9), "Densities").

This cell fits a bell curve to `balance` in each class, and then scores a customer the same way
as before, with the bell curve's height in place of the high-balance share. The other two
features still use their shares.

In [ ]:
balance_mean = Default['balance'].groupby(data['default']).mean()
balance_sd = Default['balance'].groupby(data['default']).std()
display(pd.DataFrame({'mean': balance_mean, 'standard deviation': balance_sd}).round(0))


def posterior_with_balance(student, high_income, balance):
    numerators = {}
    for k in priors.index:
        numerator = priors[k]
        numerator *= shares.loc[k, 'student'] if student else 1 - shares.loc[k, 'student']
        numerator *= shares.loc[k, 'high_income'] if high_income else 1 - shares.loc[k, 'high_income']
        numerator *= norm.pdf(balance, balance_mean[k], balance_sd[k])   # bell curve height
        numerators[k] = numerator
    return numerators['Yes'] / sum(numerators.values())


print(f'not a student, high income, balance $1,800: {posterior_with_balance(False, True, 1800):.3f}')

Now the posterior changes smoothly with the balance. Run this cell to see the two bell curves
(left) and the posterior they produce for a non-student with a high income (right). *(You don't
need to understand how this plot was generated.)*

In [ ]:
grid = np.linspace(0, 2700, 300)
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
for k, color in [('No', '#0072B2'), ('Yes', '#D55E00')]:
    in_class = Default['balance'][data['default'] == k]
    left.hist(in_class, bins=40, density=True, alpha=0.35, color=color, label=f'default = {k}')
    left.plot(grid, norm.pdf(grid, balance_mean[k], balance_sd[k]), color=color, lw=2.5)
left.set_xlabel('balance ($)')
left.set_ylabel('density')
left.set_title('Balance within each class, with fitted bell curves')
left.legend()
right.plot(grid, [posterior_with_balance(False, True, b) for b in grid], color='black', lw=2.5)
right.axhline(0.5, color='0.6', ls=':')
right.set_xlabel('balance ($)')
right.set_ylabel('posterior probability of default')
right.set_title('Not a student, high income')
fig.tight_layout()
plt.show()

The bell curves are only approximations. The non-defaulters' balances pile up at \$0, and a bell
curve can't show that pile. But the posterior now rises smoothly with the balance, and it passes
0.5 at around \$2,050, which the yes-or-no version could never reach.

**Comparing ROC curves.** With yes-or-no features, the model gave only seven different
posteriors, so its ROC curve in Step 5 had only seven corners. With the bell curve, almost every
customer gets their own posterior. Run this cell to score every customer with
`posterior_with_balance` and draw both models' ROC curves.

<details>
<summary>Code explanation</summary>

- `zip(...)` walks through the three columns together, one customer at a time, and the list
  comprehension calls `posterior_with_balance` for each customer.
- `roc_curve(is_yes, scores)` and `roc_auc_score(is_yes, scores)` take the actual classes as
  `True`/`False`, so no `pos_label` is needed.

</details>

In [ ]:
balance_posterior = pd.Series(
    [posterior_with_balance(student, high_income, balance)
     for student, high_income, balance in zip(data['student'], data['high_income'], Default['balance'])],
    index=data.index)
print('different posteriors, yes-or-no balance:', posterior.nunique())
print('different posteriors, numeric balance:  ', balance_posterior.nunique())

is_yes = data['default'] == 'Yes'
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for scores, name, color in [(posterior, 'yes-or-no balance', 'black'),
                            (balance_posterior, 'numeric balance', '#009E73')]:
    fpr, rec, _ = roc_curve(is_yes, scores)
    ax.plot(fpr, rec, color=color, lw=2.5, label=f'{name} (AUC = {roc_auc_score(is_yes, scores):.2f})')
ax.plot([0, 1], [0, 1], color='0.6', ls='--', label='random guessing')
ax.set_xlabel('false positive rate (share of non-defaulters flagged)')
ax.set_ylabel('recall (share of defaulters caught)')
ax.set_title('ROC curves: yes-or-no versus numeric balance')
ax.legend(loc='lower right')
plt.show()

With thousands of different posteriors, the numeric-balance curve is smooth, and its AUC rises
from about 0.85 to about 0.94, close to logistic regression's 0.95 in Notebook 2. Both models
use the same three columns. The difference is that the numeric model can tell a \$1,600
balance from a \$2,600 one, so it can rank customers that the yes-or-no model ties.